# TF-IDF Baseline Experiments
**Logistic Regression + Linear SVC × 7 source configs = 14 experiments**

GPU not required — runs on CPU in ~5 minutes.

Place the `data/splits/` folder under `/content/data/splits`.

## 1. Setup

In [1]:
# === Path Configuration ===
# Place the data/splits folder under /content/data/splits
SPLITS_DIR = "/content/data/splits"
RESULTS_DIR = "/content/results"
MODELS_DIR = "/content/models"

import os
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(MODELS_DIR, exist_ok=True)

# Verify data exists
files = os.listdir(SPLITS_DIR)
print(f"Found {len(files)} split files in {SPLITS_DIR}")
assert len(files) >= 21, f"Expected 21 split files, found {len(files)}"

Found 22 split files in /content/data/splits


In [2]:
import csv
import os
import pickle
import random
from collections import Counter
from datetime import datetime

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
)

# ====== Config ======
SEED = 42
LABEL_MAP = {"negative": 0, "neutral": 1, "positive": 2}
LABEL_NAMES = ["negative", "neutral", "positive"]
TFIDF_MAX_FEATURES = 50000
TFIDF_NGRAM_RANGE = (1, 2)

ALL_CONFIGS = [
    "twitter", "skytrax", "airlinequality",
    "twitter+skytrax", "twitter+airlinequality", "skytrax+airlinequality",
    "twitter+skytrax+airlinequality"
]

random.seed(SEED)
np.random.seed(SEED)
print("Setup complete.")

Setup complete.


## 2. Utility Functions

In [3]:
def load_split(source_config, split):
    """Load a split CSV and return (texts, labels)."""
    path = os.path.join(SPLITS_DIR, f"{source_config}_{split}.csv")
    texts, labels = [], []
    with open(path, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            texts.append(row["text"])
            labels.append(LABEL_MAP[row["label"]])
    return texts, labels


def compute_metrics(y_true, y_pred):
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro"),
        "macro_precision": precision_score(y_true, y_pred, average="macro"),
        "macro_recall": recall_score(y_true, y_pred, average="macro"),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted"),
    }


def compute_per_class_metrics(y_true, y_pred):
    result = {}
    for avg in ["macro", "weighted", None]:
        p = precision_score(y_true, y_pred, average=avg, zero_division=0)
        r = recall_score(y_true, y_pred, average=avg, zero_division=0)
        f = f1_score(y_true, y_pred, average=avg, zero_division=0)
        if avg is None:
            for i, name in enumerate(LABEL_NAMES):
                result[f"{name}_precision"] = p[i]
                result[f"{name}_recall"] = r[i]
                result[f"{name}_f1"] = f[i]
        else:
            result[f"{avg}_precision"] = p
            result[f"{avg}_recall"] = r
            result[f"{avg}_f1"] = f
    return result


def save_results(results_list, filename="experiment_log_tfidf.csv"):
    """Save all results to CSV."""
    path = os.path.join(RESULTS_DIR, filename)
    if not results_list:
        return
    with open(path, "w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(results_list[0].keys()))
        writer.writeheader()
        writer.writerows(results_list)
    print(f"Results saved to {path}")

print("Utilities loaded.")

Utilities loaded.


## 3. Run All 14 Experiments

In [4]:
MODEL_REGISTRY = {
    "lr": ("LogisticRegression",
           lambda: LogisticRegression(max_iter=1000, class_weight="balanced",
                                      random_state=SEED, solver="lbfgs")),
    "svc": ("LinearSVC",
            lambda: LinearSVC(max_iter=5000, class_weight="balanced",
                              random_state=SEED)),
}

all_results = []

for model_key, (model_name, model_fn) in MODEL_REGISTRY.items():
    for source_config in ALL_CONFIGS:
        print(f"\n[TRAIN] TF-IDF+{model_name} | {source_config}")

        train_texts, train_labels = load_split(source_config, "train")
        val_texts, val_labels = load_split(source_config, "val")
        test_texts, test_labels = load_split(source_config, "test")

        tfidf = TfidfVectorizer(
            max_features=TFIDF_MAX_FEATURES,
            ngram_range=TFIDF_NGRAM_RANGE,
            sublinear_tf=True,
        )
        X_train = tfidf.fit_transform(train_texts)
        X_test = tfidf.transform(test_texts)

        clf = model_fn()
        clf.fit(X_train, train_labels)
        y_pred = clf.predict(X_test)

        metrics = compute_metrics(test_labels, y_pred)
        per_class = compute_per_class_metrics(test_labels, y_pred)
        cm = confusion_matrix(test_labels, y_pred)

        print(f"  Accuracy: {metrics['accuracy']:.4f} | Macro-F1: {metrics['macro_f1']:.4f}")
        print(f"  Confusion Matrix:\n{cm}")

        row = {
            "timestamp": datetime.now().isoformat(timespec="seconds"),
            "model": f"tfidf_{model_key}",
            "source_config": source_config,
            **metrics, **per_class,
        }
        all_results.append(row)

        # Save model checkpoint
        save_dir = os.path.join(MODELS_DIR, f"tfidf_{model_key}", source_config)
        os.makedirs(save_dir, exist_ok=True)
        with open(os.path.join(save_dir, "model.pkl"), "wb") as f:
            pickle.dump(clf, f)
        with open(os.path.join(save_dir, "tfidf.pkl"), "wb") as f:
            pickle.dump(tfidf, f)

# Save all results
save_results(all_results)
print(f"\n{'='*60}")
print(f"DONE: {len(all_results)} experiments completed.")


[TRAIN] TF-IDF+LogisticRegression | twitter
  Accuracy: 0.7925 | Macro-F1: 0.7357
  Confusion Matrix:
[[1160  142   51]
 [ 114  269   46]
 [  47   37  240]]

[TRAIN] TF-IDF+LogisticRegression | skytrax
  Accuracy: 0.7919 | Macro-F1: 0.6843
  Confusion Matrix:
[[1556  232   96]
 [ 243  252  158]
 [ 131  290 2569]]

[TRAIN] TF-IDF+LogisticRegression | airlinequality
  Accuracy: 0.7885 | Macro-F1: 0.6175
  Confusion Matrix:
[[1997  127  289]
 [  66   68   92]
 [  78   56  574]]

[TRAIN] TF-IDF+LogisticRegression | twitter+skytrax
  Accuracy: 0.7815 | Macro-F1: 0.7096
  Confusion Matrix:
[[2644  449  144]
 [ 331  524  227]
 [ 187  330 2797]]

[TRAIN] TF-IDF+LogisticRegression | twitter+airlinequality
  Accuracy: 0.7612 | Macro-F1: 0.6739
  Confusion Matrix:
[[2964  414  388]
 [ 148  377  130]
 [ 125   97  810]]

[TRAIN] TF-IDF+LogisticRegression | skytrax+airlinequality
  Accuracy: 0.7870 | Macro-F1: 0.6926
  Confusion Matrix:
[[3458  431  408]
 [ 252  424  203]
 [ 199  397 3102]]

[TRAIN

## 4. Results Summary

In [5]:
print(f"{'Model':<15} {'Source Config':<35} {'Accuracy':>10} {'Macro-F1':>10}")
print("=" * 75)
for r in all_results:
    print(f"{r['model']:<15} {r['source_config']:<35} {r['accuracy']:>10.4f} {r['macro_f1']:>10.4f}")

# Best model
best = max(all_results, key=lambda x: x["macro_f1"])
print(f"\nBest: {best['model']} | {best['source_config']} | Macro-F1: {best['macro_f1']:.4f}")

Model           Source Config                         Accuracy   Macro-F1
tfidf_lr        twitter                                 0.7925     0.7357
tfidf_lr        skytrax                                 0.7919     0.6843
tfidf_lr        airlinequality                             0.7885     0.6175
tfidf_lr        twitter+skytrax                         0.7815     0.7096
tfidf_lr        twitter+airlinequality                     0.7612     0.6739
tfidf_lr        skytrax+airlinequality                     0.7870     0.6926
tfidf_lr        twitter+skytrax+airlinequality             0.7748     0.6999
tfidf_svc       twitter                                 0.8048     0.7303
tfidf_svc       skytrax                                 0.8127     0.6617
tfidf_svc       airlinequality                             0.8034     0.5746
tfidf_svc       twitter+skytrax                         0.7955     0.6977
tfidf_svc       twitter+airlinequality                     0.7799     0.6582
tfidf_svc       skyt

## 5. Download Results

In [6]:
import subprocess
subprocess.run(["zip", "-r", "/content/results_tfidf.zip", "/content/results", "/content/models"], check=True)
print("Zip created: /content/results_tfidf.zip")
print("Download this file from the file explorer.")

Zip created: /content/results_tfidf.zip
Download this file from the file explorer.
